# Week 46, Day 2: Beyond persistent activity: dynamic and activity-silent memory

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w046/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Stable and dynamic codes

Write ctd(X, lab): split the trials in half at random; for each training time, take the difference of class means as the decoder; test it on the other half at every time by the sign of the projection. Apply it to three simulated populations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2015)
N, T, n = 80, 20, 100

def make(kind):
    m1, m2, ms = r.normal(size=(3, N))
    pats = []
    for t in range(T):
        ph = 2 * np.pi * t / T * 1.5
        dyn = np.cos(ph) * m1 + np.sin(ph) * m2
        pats.append({"stable": ms, "dynamic": dyn, "mixed": 0.6 * ms + 0.8 * dyn}[kind])
    pats = 0.25 * np.array(pats)
    lab = np.r_[np.ones(n), -np.ones(n)]
    X = lab[:, None, None] * pats[None] + r.normal(size=(2 * n, T, N))
    return X, lab

def ctd(X, lab):
    idx = r.permutation(len(lab)); tr, te = idx[: len(lab) // 2], idx[len(lab) // 2 :]
    A = np.zeros((T, T))
    for t1 in range(T):
        w = ...
        for t2 in range(T):
            A[t1, t2] = ...
    return A

def far(A):
    gap = np.abs(np.subtract.outer(np.arange(T), np.arange(T)))
    return A[gap >= 8].mean()

data = {k: make(k) for k in ["stable", "dynamic", "mixed"]}
M = {k: ctd(*data[k]) for k in data}
fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
for a_, k in zip(ax, M):
    a_.imshow(M[k], vmin=0.5, vmax=1, origin="lower"); a_.set_title(f"{k}: far {far(M[k]):.2f}"); a_.set_xlabel("test time"); a_.set_ylabel("train time")
plt.tight_layout(); plt.show()
print("mean accuracy for gaps of 8 or more bins:", ", ".join(f"{k} {far(v):.2f}" for k, v in M.items()))

In [ ]:
import numpy as np
def test_diag():
    "every code is decodable at every time (diagonal accuracy above 0.95)"
    assert all(np.diag(M[k]).mean() > 0.95 for k in M)
def test_stable():
    "the stable code generalises across time (accuracy above 0.95 for gaps of 8 bins or more)"
    assert far(M["stable"]) > 0.95
def test_dynamic():
    "the dynamic code does not generalise (below 0.65 for long gaps)"
    assert far(M["dynamic"]) < 0.65

test_diag()
test_stable()
test_dynamic()
print("All checks passed")

### Exercise: Finding the stable part

For the mixed population, build w_avg from the class-mean difference averaged over all time bins of the training half, and w_first from time 0 only. Test both at every time on the other half.

In [ ]:
X, lab = data["mixed"]
idx = r.permutation(len(lab)); tr, te = idx[:n], idx[n:]
diff = X[tr][lab[tr] > 0].mean(0) - X[tr][lab[tr] < 0].mean(0)     # T x N class difference
w_avg = ...
w_first = ...
acc_avg = np.array([np.mean(np.sign(X[te][:, t] @ w_avg) == lab[te]) for t in range(T)])
acc_first = np.array([np.mean(np.sign(X[te][:, t] @ w_first) == lab[te]) for t in range(T)])
print(f"time-averaged decoder: min {acc_avg.min():.2f}, mean {acc_avg.mean():.2f};  first-bin decoder: mean {acc_first.mean():.2f}")
plt.plot(acc_avg, label="time-averaged decoder"); plt.plot(acc_first, label="trained at t = 0"); plt.ylim(0.4, 1.02); plt.xlabel("test time bin"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_general():
    "the time-averaged decoder exceeds 0.75 accuracy at every time bin of the mixed code"
    assert acc_avg.min() > 0.75
def test_better():
    "it beats a decoder trained at the first time bin, averaged over all test times"
    assert acc_avg.mean() > acc_first.mean() + 0.1

test_general()
test_better()
print("All checks passed")

### Exercise: Reading out a silent memory

For each delay, compute the facilitation of the cued pool, U + (1 - U) exp(-delay / tauF), and of the other pool (U). Simulate ping responses proportional to facilitation plus noise for 400 trials, decode the item as the pool with the larger response, and record accuracy.

In [ ]:
tauF, U0, noise = 1.5, 0.2, 0.3
delays = np.array([0.5, 1, 2, 4, 8])
acc = []
for d in delays:
    u_cued = ...
    item = r.integers(0, 2, 400)
    resp = np.full((400, 2), U0)
    resp[np.arange(400), item] = u_cued
    resp = resp + noise * r.normal(size=resp.shape)
    acc.append(...)
acc = np.array(acc)
print("decoding accuracy from the ping response:", ", ".join(f"{d:g} s: {a_:.2f}" for d, a_ in zip(delays, acc)))

In [ ]:
import numpy as np
def test_early():
    "half a second after the cue, the ping response decodes the item above 85%"
    assert acc[0] > 0.85
def test_decay():
    "decoding falls towards chance as facilitation decays"
    assert all(np.diff(acc) <= 0.02) and acc[-1] < 0.6

test_early()
test_decay()
print("All checks passed")